In [1]:
from IPython.display import display, HTML
display(HTML("""
<style>
div.container{width:95% !important;}
div.cell.code_cell.rendered{width:95%;}
div.input_prompt{padding:0px;}
div.CodeMirror {font-family:Consolas; font-size:22pt;}
.inner_cell{font-size:22pt;}
div.text_cell_render pre code {font-size:22pt; line-height:30px;}
div.output {font-size:20pt; font-weight:bold;}
div.input {font-family:Consolas; font-size:22pt;}
div.prompt {min-width:70px;}
div#toc-wrapper{padding-top:120px;}
div.text_cell_render li{font-size:20pt;padding:5px; line-height:30px;}
table.dataframe{font-size:22px;}
</style>
"""))

**<font size="6" color="red">ch8. Attention추가(스마트 번역기)</font>**
- Google Neural Machine Translation(GNMT)
- RNN기반의 Seq2Seq방식
- 인코더입력/디코더입력(모범답안) - 디코더 출력(답안) ; 인코더와 디코더가 연결된 구조

# 1. 패키지 import 및 하이퍼 파라미터

In [1]:
import numpy as np
import pandas as pd
from time import time

from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical

# 하이퍼 파라미터
MY_HIDDEN = 128
MY_EPOCH = 500

# 2. 학습데이터

In [2]:
raw = pd.read_csv('data/translate.csv', header=None)
eng_kor = raw.values.tolist() # 데이터프레임을 list로 변환
print(eng_kor[:3])
print('학습할 영-한 데이터 갯수 :', len(eng_kor))

[['cold', '감기'], ['come', '오다'], ['cook', '요리']]
학습할 영-한 데이터 갯수 : 110


In [3]:
e_alpha = [c for c in 'SEPabcdefghijklmnopqrstuvwxyz']
korean = ''.join([data[1] for data in eng_kor])
k_alpha = list(set([ch for ch in korean]))
k_alpha.sort()
print(k_alpha)

['가', '각', '간', '감', '개', '거', '것', '게', '계', '고', '관', '광', '구', '굴', '규', '그', '금', '기', '깊', '나', '날', '남', '내', '넓', '녀', '노', '놀', '농', '높', '뉴', '늦', '다', '단', '도', '동', '들', '람', '랑', '래', '램', '류', '름', '릎', '리', '많', '망', '매', '머', '먼', '멍', '메', '명', '모', '목', '무', '물', '미', '바', '반', '방', '번', '복', '부', '분', '붕', '비', '뿌', '사', '상', '색', '생', '서', '선', '소', '손', '수', '쉽', '스', '시', '식', '실', '싸', '아', '약', '얇', '어', '언', '얼', '여', '연', '오', '옥', '왼', '요', '용', '우', '운', '움', '위', '유', '은', '을', '음', '의', '이', '익', '인', '읽', '입', '자', '작', '장', '적', '제', '좋', '주', '지', '짜', '쪽', '찾', '책', '출', '칙', '크', '키', '탈', '택', '통', '파', '팔', '편', '피', '핑', '한', '합', '해', '행', '험', '회', '획', '휴', '흐']


In [4]:
alpha = e_alpha + k_alpha
print('영어와 한글 알파벳 :', alpha)
alpha_total_size = len(alpha)
print('전체 알파벳 갯수(원핫인코딩 사이즈) :', alpha_total_size)

영어와 한글 알파벳 : ['S', 'E', 'P', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z', '가', '각', '간', '감', '개', '거', '것', '게', '계', '고', '관', '광', '구', '굴', '규', '그', '금', '기', '깊', '나', '날', '남', '내', '넓', '녀', '노', '놀', '농', '높', '뉴', '늦', '다', '단', '도', '동', '들', '람', '랑', '래', '램', '류', '름', '릎', '리', '많', '망', '매', '머', '먼', '멍', '메', '명', '모', '목', '무', '물', '미', '바', '반', '방', '번', '복', '부', '분', '붕', '비', '뿌', '사', '상', '색', '생', '서', '선', '소', '손', '수', '쉽', '스', '시', '식', '실', '싸', '아', '약', '얇', '어', '언', '얼', '여', '연', '오', '옥', '왼', '요', '용', '우', '운', '움', '위', '유', '은', '을', '음', '의', '이', '익', '인', '읽', '입', '자', '작', '장', '적', '제', '좋', '주', '지', '짜', '쪽', '찾', '책', '출', '칙', '크', '키', '탈', '택', '통', '파', '팔', '편', '피', '핑', '한', '합', '해', '행', '험', '회', '획', '휴', '흐']
전체 알파벳 갯수(원핫인코딩 사이즈) : 171


# 3. 문자당 num을 갖는 dict
- 전예제 : {'the':1, 'a':2,...} / {1:'the', 2:'a', ...}
- {'S':0, 'E':1, ...}

In [5]:
# char_to_num = {}
# for i, ch in enumerate(alpha):
#     #print(i, ch)
#     char_to_num[ch] = i
char_to_num = { ch:i for i, ch in enumerate(alpha)}
print(char_to_num)

{'S': 0, 'E': 1, 'P': 2, 'a': 3, 'b': 4, 'c': 5, 'd': 6, 'e': 7, 'f': 8, 'g': 9, 'h': 10, 'i': 11, 'j': 12, 'k': 13, 'l': 14, 'm': 15, 'n': 16, 'o': 17, 'p': 18, 'q': 19, 'r': 20, 's': 21, 't': 22, 'u': 23, 'v': 24, 'w': 25, 'x': 26, 'y': 27, 'z': 28, '가': 29, '각': 30, '간': 31, '감': 32, '개': 33, '거': 34, '것': 35, '게': 36, '계': 37, '고': 38, '관': 39, '광': 40, '구': 41, '굴': 42, '규': 43, '그': 44, '금': 45, '기': 46, '깊': 47, '나': 48, '날': 49, '남': 50, '내': 51, '넓': 52, '녀': 53, '노': 54, '놀': 55, '농': 56, '높': 57, '뉴': 58, '늦': 59, '다': 60, '단': 61, '도': 62, '동': 63, '들': 64, '람': 65, '랑': 66, '래': 67, '램': 68, '류': 69, '름': 70, '릎': 71, '리': 72, '많': 73, '망': 74, '매': 75, '머': 76, '먼': 77, '멍': 78, '메': 79, '명': 80, '모': 81, '목': 82, '무': 83, '물': 84, '미': 85, '바': 86, '반': 87, '방': 88, '번': 89, '복': 90, '부': 91, '분': 92, '붕': 93, '비': 94, '뿌': 95, '사': 96, '상': 97, '색': 98, '생': 99, '서': 100, '선': 101, '소': 102, '손': 103, '수': 104, '쉽': 105, '스': 106, '시': 107, '식': 108, '실': 109, '싸': 110,

In [14]:
# 문자->숫자 / 숫자->문자
print('문자->숫자 : ',char_to_num.get('c', -1))
print('숫자->문자 : ',alpha[5])

문자->숫자 :  5
숫자->문자 :  c


In [7]:
data = eng_kor[0]
print(data)
print('인코더 입력(원핫인코딩 전) :', [char_to_num.get(ch) for ch in data[0]])
print('디코더 입력(원핫인코딩 전) :', [char_to_num.get(ch) for ch in 'S'+data[1]])
print('디코더 출력 :', [char_to_num.get(ch) for ch in data[1]+'E'])

['cold', '감기']
인코더 입력(원핫인코딩 전) : [5, 17, 14, 6]
디코더 입력(원핫인코딩 전) : [0, 32, 46]
디코더 출력 : [32, 46, 1]


In [24]:
# 원핫인코딩 방법1 : 이 코드에서는 불가
pd.get_dummies([5, 17, 14, 6])

,5,6,14,17
0,1,0,0,0
1,0,0,0,1
2,0,0,1,0
3,0,1,0,0


In [ ]:
# 원핫인코딩 방법2
to_categorical([5, 17, 14, 6], num_classes=alpha_total_size)

In [32]:
# 원핫인코딩 방법3 : np.eye(n) - n행n열 단위행렬(A@단위행렬=단위행렬@A=A)
np.eye(alpha_total_size)[[5, 17, 14, 6]]

# 4. 인코더입력, 디코더입력, 디코더출력
- 인코더입력과 디코더입력(원핫인코딩O), 디코더출력(원핫인코딩X-loss를 sparse categoricalcrossentropy)

In [33]:
eng_kor[:3]

[['cold', '감기'], ['come', '오다'], ['cook', '요리']]

In [6]:
def encoding(eng_kor=eng_kor):
    '인코더입력데이터(110*4*171), 디코더입력데이터(110*3*171), 디코더출력데이터(110,3,1)를 return'
    enc_in = [] # 인코더입력(cold 원핫인코딩)
    dec_in = [] # 디코더입력(S감기 원핫인코딩)
    dec_out = [] # 디코더출력(감기E 라벨인코딩)
    for data in eng_kor:
        # 인코더 입력(영어 -> 숫자 -> 원핫코딩)
        eng = [char_to_num.get(ch) for ch in data[0]]
        eng_one = to_categorical(eng, num_classes=alpha_total_size)
        enc_in.append(eng_one)
        # 디코더 입력('S'한글 -> 숫자 -> 원핫인코딩 )
        kor = [char_to_num.get(ch) for ch in 'S'+data[1]]
        kor_one = np.eye(alpha_total_size)[kor]
        dec_in.append(kor_one)
        # 디코더 출력( 한글'E' -> 숫자)
        kor = [[char_to_num.get(ch)] for ch in data[1]+'E']
        dec_out.append(kor)
        #print(kor)
    # 인공신경망에 넣을 데이터이므로 numpy 배열로 전환
    enc_in = np.array(enc_in)
    dec_in = np.array(dec_in)
    dec_out = np.array(dec_out)
    # print(enc_in.shape, dec_in.shape, dec_out.shape)
    return enc_in, dec_in, dec_out
        
sample = [['cold', '감기'], ['come', '오다']]
X_enc, X_dec, Y_dec = encoding(sample)
X_enc.shape, X_dec.shape, Y_dec.shape

((2, 4, 171), (2, 3, 171), (2, 3, 1))

# 5. 전체 번역데이터(독립변수, 타겟변수)

In [7]:
# Seq2Seq에 들어갈 데이터
X_enc, X_dec, Y_dec = encoding(eng_kor)
X_enc.shape, X_dec.shape, Y_dec.shape

((110, 4, 171), (110, 3, 171), (110, 3, 1))

# 6. 모델구현(Seq2Seq모델)
- 교안pdf 119p

In [59]:
# 인코더 LSTM 구현
ENC_IN = Input(shape=(4, alpha_total_size))
_, state_h, state_c = LSTM(units=MY_HIDDEN,
                          return_state=True, # 디코더 입력을 받기 위한 
                          # return_sequences=False 기본값
                          )(ENC_IN)
# 인코더와 디코더를 연결할 link
link = [state_h, state_c]
# 디코더 LSTM 구현
DEC_IN = Input(shape=(3, alpha_total_size))
DEC_MID = LSTM(units=MY_HIDDEN,
            # return_state=False, 기본값
            return_sequences=True
            )(DEC_IN, initial_state=link)
# 최종 출력 층
DEC_OUT = Dense(units=alpha_total_size, activation='softmax')(DEC_MID)
# 모델
model = Model(inputs=[ENC_IN, DEC_IN], 
             outputs=DEC_OUT,
             name='model')
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_10 (InputLayer)          [(None, 4, 171)]     0           []                               
                                                                                                  
 input_11 (InputLayer)          [(None, 3, 171)]     0           []                               
                                                                                                  
 lstm_9 (LSTM)                  [(None, 128),        153600      ['input_10[0][0]']               
                                 (None, 128),                                                     
                                 (None, 128)]                                                     
                                                                                              

# 6. 모델구현(Attention추가)
- 두번째 교안 pdf 26p

In [13]:
from tensorflow.keras.layers import Attention, Concatenate
# 인코더 LSTM 구현
ENC_IN = Input(shape=(4, alpha_total_size))
ENC_OUT, state_h, state_c = LSTM(units=MY_HIDDEN,
                                return_sequences=True, # 위 출력
                                return_state=True)(ENC_IN) # h와 c 옆 출력
# 디코더 LSTM 구현
DEC_IN = Input(shape=(3, alpha_total_size))
DEC_MID, _, _ = LSTM(units=MY_HIDDEN,
                    return_sequences=True,
                    return_state=True)(DEC_IN,
                                      initial_state=[state_h, state_c])
# 어텐션 메커니즘
CONTEXT_VECTOR = Attention()([DEC_MID, ENC_OUT])
# CONTEXT_VECTOR와 디코더LSTM 출력을 결합
CONTEXT_AND_LSTM = Concatenate()([CONTEXT_VECTOR, DEC_MID])
# 최종 출력층
OUT = Dense(units=alpha_total_size, activation='softmax')(CONTEXT_AND_LSTM)
# 모델
model = Model(inputs = [ENC_IN, DEC_IN],
             outputs = OUT,
             name='model')
model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_8 (InputLayer)           [(None, 4, 171)]     0           []                               
                                                                                                  
 input_9 (InputLayer)           [(None, 3, 171)]     0           []                               
                                                                                                  
 lstm_7 (LSTM)                  [(None, 4, 128),     153600      ['input_8[0][0]']                
                                 (None, 128),                                                     
                                 (None, 128)]                                                     
                                                                                              

# 7. 모델 학습

In [14]:
model.compile(loss='sparse_categorical_crossentropy',
             optimizer='rmsprop',
             metrics=['accuracy'])
begin = time()
hist = model.fit([X_enc, X_dec],
                Y_dec,
                epochs=MY_EPOCH,
                verbose=0)
end = time()
print(f'학습시간 : {end-begin:.2f}초')

학습시간 : 22.11초


In [15]:
model.evaluate([X_enc, X_dec], Y_dec)

4/4 [==============================] - 1s 6ms/step - loss: 3.6630e-07 - accuracy: 1.0000


[3.662975416318659e-07, 1.0]

# 8. 모델 사용

In [16]:
# 쉬운 문제
easy_test = [['cold', 'pp'],
             ['down', 'pp'],
             ['desk', 'pp'],
             ['love', 'pp'],
             ['wood', 'pp']]
enc_in, dec_in, _ = encoding(easy_test)
enc_in.shape, dec_in.shape, _.shape

((5, 4, 171), (5, 3, 171), (5, 3, 1))

In [17]:
pred = model.predict([enc_in, dec_in])
pred.argmax(axis=-1)

1/1 [==============================] - 1s 580ms/step


array([[ 32,  46,   1],
       [111,  67,   1],
       [149,  97,   1],
       [ 96,  66,   1],
       [ 48,  83,   1]], dtype=int64)

In [18]:
for i in range(len(pred)):
    eng = easy_test[i][0]
    hat = pred[i].argmax(axis=-1)
    kor = ''.join([alpha[num] for num in hat[:-1]])
    print(f'{eng} => {kor}{hat[:-1]}')

cold => 감기[32 46]
down => 아래[111  67]
desk => 책상[149  97]
love => 사랑[96 66]
wood => 나무[48 83]


In [19]:
# 어려운 문제
easy_test = [['lvoe', 'pp'],
             ['loev', 'pp'],
             ['olve', 'pp'],
             ['love', 'pp'],
             ['eovl', 'pp']]
enc_in, dec_in, _ = encoding(easy_test)
enc_in.shape, dec_in.shape, _.shape

((5, 4, 171), (5, 3, 171), (5, 3, 1))

In [20]:
pred = model.predict([enc_in, dec_in])
pred.argmax(axis=-1)

1/1 [==============================] - 0s 22ms/step


array([[96, 66,  1],
       [96, 66,  1],
       [96, 66,  1],
       [96, 66,  1],
       [60, 60,  1]], dtype=int64)

In [21]:
for i in range(len(pred)):
    eng = easy_test[i][0]
    hat = pred[i].argmax(axis=-1)
    kor = ''.join([alpha[num] for num in hat[:-1]])
    print(f'{eng} => {kor}{hat[:-1]}')

lvoe => 사랑[96 66]
loev => 사랑[96 66]
olve => 사랑[96 66]
love => 사랑[96 66]
eovl => 다다[60 60]
